In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import pandas as pd
import config
import os
import config
# Entrenamiento (Noviembre)
from src.modelling.transforms import EngagementScaler
from src.modelling.similarity import KendallTauSimilarity
from src.modelling.spectral import SpectralClusteringModel

In [ ]:
cols = config.FEATURES_GENERIC
workpath = config.WORKPATH
n_clusters = config.CLUSTER_GENERIC


df_train = pd.read_parquet(os.path.join(workpath, "processed", "training_201911", "aggregated_generic.parquet"))
df_test = pd.read_parquet(os.path.join(workpath, "processed", "validation_201912", "aggregated_generic.parquet"))

# 1. Ranking y normalización
scaler = EngagementScaler(features=cols)
df_ranked = scaler.fit(df_train)  # Calcula means y stds
df_normalized = scaler.transform(df_ranked)
df_normalized['cluster'] = None  # placeholder

# 2. Similitud y afinidad
similarity = KendallTauSimilarity(method='kendall')
affinity_matrix = similarity.fit_transform(df_normalized, features=cols)

# 3. Clustering y guardar artefactos
spectral_model = SpectralClusteringModel(n_clusters=3, n_init=50)
labels = spectral_model.fit(affinity_matrix, df_normalized, features=cols)

# Guardar estadísticas de entrenamiento
spectral_model.set_training_statistics(df_ranked, features=cols)

# Obtener resultados
df_clusters = spectral_model.get_clusters_dataframe(df_normalized)

# Guardar artefactos
spectral_model.save_model_artifacts('models/nov_generic_artifacts.npy')


# ============= PROYECCIÓN (Diciembre) =============

# 1. Cargar estadísticas de entrenamiento
scaler_dec = EngagementScaler(features=cols)
scaler_dec.means = spectral_model.train_rank_means
scaler_dec.stds = spectral_model.train_rank_stds

# 2. Aplicar transformación (sin reajustar)
df_ranked_dec = df_test.copy()
df_ranked_dec[cols] = df_ranked_dec[cols].rank(method='dense')
df_normalized_dec = scaler_dec.transform(df_ranked_dec)

# 3. Proyectar en espacio de centroides
df_normalized_dec["cluster"] = spectral_model.predict(df_normalized_dec[cols].values)

# 2. Similitud y afinidad
similarity = KendallTauSimilarity(method='kendall')
affinity_matrix_dec = similarity.fit_transform(df_normalized_dec, features=cols)